# Week 3 - Semantic RAG Orchestration with FastAPI

This notebook is regenerated for **Week 3** and is designed to work on top of **Week 1** and **Week 2** outputs while giving stronger semantic retrieval quality.

## What this notebook does
- loads processed chunk data and metadata from the required repository structure
- loads or rebuilds a semantic vector index
- applies metadata-aware retrieval
- optionally reranks retrieved chunks for better relevance
- integrates an LLM with grounded prompting
- exposes FastAPI endpoints
- evaluates answer quality and latency

## Important alignment with your 3-week plan
- **Week 1:** data ingestion, metadata, chunking, validation
- **Week 2:** embeddings, vector store, semantic retrieval, filtering, evaluation
- **Week 3:** LLM grounding, RAG orchestration, API, testing

## Repository structure expected
```text
hr-compliance-rag/
├── data/
│   ├── raw/
│   ├── processed/
│   └── metadata.csv
├── ingestion/
│   ├── loaders.py
│   ├── chunker.py
│   └── validator.py
├── notebooks/
├── api/
├── vectorstore/
├── README.md
└── requirements.txt
```

## Best-results strategy used here
- stronger semantic embedding model
- metadata-preserving retrieval
- optional cross-encoder reranking
- strict grounded prompting
- reproducible configuration
- latency and failure-case evaluation

## 1. Install Required Libraries

In [ ]:
!pip -q install sentence-transformers faiss-cpu transformers accelerate fastapi uvicorn pydantic nest-asyncio pandas numpy pyarrow scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 73.3 MB/s eta 0:00:00


## 2. Imports and Configuration

In [ ]:
import os
import pandas as pd

# Read files separately if needed
metadata_df = pd.read_csv("/content/metadata.csv")
metadata_chunks_df = pd.read_csv("/content/metadata_with_chunks.csv")
document_metadata_df = pd.read_csv("/content/document_metadata.csv")

REQUIRED_METADATA_FIELDS = [
    "department",
    "document_type",
    "category",
    "region",
    "year",
    "source",
    "file_name"
]

CONFIG = {
    "processed_candidates": [
        "/content/metadata_with_chunks.csv",
        "/content/processed_chunks.csv",
        "/content/processed_chunks.parquet"
    ]
}

def find_processed_file(candidates):
    for path in candidates:
        if os.path.exists(path):
            return path
    return None

def load_processed_chunks():
    processed_path = find_processed_file(CONFIG["processed_candidates"])
    if processed_path is None:
        raise FileNotFoundError(
            "No processed chunk file found. Expected one of: " + ", ".join(CONFIG["processed_candidates"])
        )

    if processed_path.endswith(".parquet"):
        frame = pd.read_parquet(processed_path)
    else:
        frame = pd.read_csv(processed_path)

    frame.columns = [c.strip().lower() for c in frame.columns]

    if "chunk" not in frame.columns and "text" in frame.columns:
        frame = frame.rename(columns={"text": "chunk"})

    if "chunk_id" not in frame.columns:
        frame["chunk_id"] = [f"chunk_{i}" for i in range(len(frame))]

    for col in REQUIRED_METADATA_FIELDS:
        if col not in frame.columns:
            frame[col] = "unknown"

    if "chunk" not in frame.columns:
        raise ValueError("The processed file must contain either 'chunk' or 'text' column.")

    frame["chunk"] = frame["chunk"].astype(str).fillna("").str.strip()
    frame = frame[frame["chunk"] != ""].reset_index(drop=True)

    return frame, processed_path

chunks_df, processed_path_used = load_processed_chunks()

print("Processed chunks loaded from:", processed_path_used)
print("Rows:", len(chunks_df))
print("Columns:", list(chunks_df.columns))
print(chunks_df.head(3))

Processed chunks loaded from: /content/metadata_with_chunks.csv
Rows: 2129
Columns: ['chunk_id', 'chunk', 'chunk_tokens', 'department', 'document_type', 'category', 'region', 'year', 'source', 'file_name']
            chunk_id                                              chunk  \
0  ag_news_0_chunk_1  Wall St. Bears Claw Back Into the Black (Reute...   
1  ag_news_1_chunk_1  Carlyle Looks Toward Commercial Aerospace (Reu...   
2  ag_news_2_chunk_1  Oil and Economy Cloud Stocks' Outlook (Reuters...   

   chunk_tokens       department document_type    category  region  year  \
0            36  Human Resources  Policy/Legal  Compliance  Global  2020   
1            54  Human Resources  Policy/Legal  Compliance  Global  2021   
2            50  Human Resources  Policy/Legal  Compliance  Global  2022   

    source      file_name  
0  ag_news  ag_news_0.txt  
1  ag_news  ag_news_1.txt  
2  ag_news  ag_news_2.txt  


## 3. Week 3 Design Notes

### Why this notebook gives better semantic results
- `all-mpnet-base-v2` is used for stronger semantic retrieval than smaller baseline models
- retrieved chunks keep **mandatory metadata traceability**
- optional **CrossEncoder reranking** improves top context quality before generation
- the LLM prompt is strictly grounded and forbids outside knowledge
- the answer returns supporting citations from retrieved chunks

### Week 3 rule followed
This notebook does **not bypass retrieval** before generating answers.

## 4. Load Week 1 and Week 2 Outputs

This notebook expects the Week 1 and Week 2 repository outputs to exist.

### Required minimum inputs
- `data/metadata.csv`
- one processed chunks file inside `data/processed/`
- optional existing vector store from Week 2

If the FAISS index is missing but processed chunks are available, the notebook can rebuild the index.

In [ ]:
import os
import pandas as pd

# =========================
# 1. Configuration
# =========================
REQUIRED_METADATA_FIELDS = [
    "department",
    "document_type",
    "category",
    "region",
    "year",
    "source",
    "file_name"
]

# Use the file that already contains chunks as the first preference
CONFIG = {
    "processed_candidates": [
        "/content/metadata_with_chunks.csv",
        "/content/metadata.csv",
        "/content/document_metadata.csv"
    ]
}

# =========================
# 2. Helper function to find the first available processed file
# =========================
def find_processed_file(candidates):
    for path in candidates:
        if os.path.exists(path):
            return path
    return None

# =========================
# 3. Load and clean processed chunks
# =========================
def load_processed_chunks():
    processed_path = find_processed_file(CONFIG["processed_candidates"])

    if processed_path is None:
        raise FileNotFoundError(
            "No processed file found. Expected one of:\n" + "\n".join(CONFIG["processed_candidates"])
        )

    # Load file
    if processed_path.endswith(".parquet"):
        frame = pd.read_parquet(processed_path)
    else:
        frame = pd.read_csv(processed_path)

    # Standardize column names
    frame.columns = [c.strip().lower() for c in frame.columns]

    # Rename text -> chunk if needed
    if "chunk" not in frame.columns and "text" in frame.columns:
        frame = frame.rename(columns={"text": "chunk"})

    # Create chunk_id if missing
    if "chunk_id" not in frame.columns:
        frame["chunk_id"] = [f"chunk_{i}" for i in range(len(frame))]

    # Add missing metadata columns
    for col in REQUIRED_METADATA_FIELDS:
        if col not in frame.columns:
            frame[col] = "unknown"

    # Validate chunk column
    if "chunk" not in frame.columns:
        raise ValueError(
            f"The loaded file '{processed_path}' does not contain a 'chunk' column."
        )

    # Clean chunk column
    frame["chunk"] = frame["chunk"].fillna("").astype(str).str.strip()
    frame = frame[frame["chunk"] != ""].reset_index(drop=True)

    # Clean metadata columns
    for col in REQUIRED_METADATA_FIELDS:
        frame[col] = frame[col].fillna("unknown").astype(str).str.strip()

    return frame, processed_path

# =========================
# 4. Load data
# =========================
df=pd.read_csv("/content/metadata.csv")
df=pd.read_csv("/content/document_metadata.csv")
df=pd.read_csv("/content/metadata_with_chunks.csv")

# =========================
# 5. Output checks
# =========================
print("Processed chunks loaded from:", processed_path_used)
print("Number of rows:", len(chunks_df))
print("Columns:", list(chunks_df.columns))

print("\nFirst 3 rows:")
display(chunks_df.head(3))

print("\nMissing values per required metadata field:")
display(chunks_df[REQUIRED_METADATA_FIELDS].isnull().sum())

print("\nSample chunk:")
print(chunks_df.loc[0, "chunk"][:500] if len(chunks_df) > 0 else "No chunk data available.")

Processed chunks loaded from: /content/metadata_with_chunks.csv
Number of rows: 2129
Columns: ['chunk_id', 'chunk', 'chunk_tokens', 'department', 'document_type', 'category', 'region', 'year', 'source', 'file_name']

First 3 rows:


,chunk_id,chunk,chunk_tokens,department,document_type,category,region,year,source,file_name
0,ag_news_0_chunk_1,Wall St. Bears Claw Back Into the Black (Reute...,36,Human Resources,Policy/Legal,Compliance,Global,2020,ag_news,ag_news_0.txt
1,ag_news_1_chunk_1,Carlyle Looks Toward Commercial Aerospace (Reu...,54,Human Resources,Policy/Legal,Compliance,Global,2021,ag_news,ag_news_1.txt
2,ag_news_2_chunk_1,Oil and Economy Cloud Stocks' Outlook (Reuters...,50,Human Resources,Policy/Legal,Compliance,Global,2022,ag_news,ag_news_2.txt



Missing values per required metadata field:


,0
department,0
document_type,0
category,0
region,0
year,0
source,0
file_name,0



Sample chunk:
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.


In [ ]:
REQUIRED_METADATA_FIELDS = [
    "department",
    "document_type",
    "category",
    "region",
    "year",
    "source",
    "file_name"
]

def find_processed_file(candidates):
    for path in candidates:
        if os.path.exists(path):
            return path
    return None

def load_processed_chunks():
    processed_path = find_processed_file(CONFIG["processed_candidates"])
    if processed_path is None:
        raise FileNotFoundError(
            "No processed chunk file found. Expected one of: " + ", ".join(CONFIG["processed_candidates"])
        )

    if processed_path.endswith(".parquet"):
        frame = pd.read_parquet(processed_path)
    else:
        frame = pd.read_csv(processed_path)

    frame.columns = [c.strip().lower() for c in frame.columns]

    if "chunk" not in frame.columns and "text" in frame.columns:
        frame = frame.rename(columns={"text": "chunk"})
    if "chunk_id" not in frame.columns:
        frame["chunk_id"] = [f"chunk_{i}" for i in range(len(frame))]

    for col in REQUIRED_METADATA_FIELDS:
        if col not in frame.columns:
            frame[col] = "unknown"

    frame["chunk"] = frame["chunk"].astype(str).fillna("").str.strip()
    frame = frame[frame["chunk"] != ""].reset_index(drop=True)

    return frame, processed_path

chunks_df, processed_path_used = load_processed_chunks()

print("Processed chunks loaded from:", processed_path_used)
print("Rows:", len(chunks_df))
print("Columns:", list(chunks_df.columns))
chunks_df.head(3)

Processed chunks loaded from: /content/metadata_with_chunks.csv
Rows: 2129
Columns: ['chunk_id', 'chunk', 'chunk_tokens', 'department', 'document_type', 'category', 'region', 'year', 'source', 'file_name']


,chunk_id,chunk,chunk_tokens,department,document_type,category,region,year,source,file_name
0,ag_news_0_chunk_1,Wall St. Bears Claw Back Into the Black (Reute...,36,Human Resources,Policy/Legal,Compliance,Global,2020,ag_news,ag_news_0.txt
1,ag_news_1_chunk_1,Carlyle Looks Toward Commercial Aerospace (Reu...,54,Human Resources,Policy/Legal,Compliance,Global,2021,ag_news,ag_news_1.txt
2,ag_news_2_chunk_1,Oil and Economy Cloud Stocks' Outlook (Reuters...,50,Human Resources,Policy/Legal,Compliance,Global,2022,ag_news,ag_news_2.txt


## 5. Validate Metadata Completeness

Week 1 required metadata is carried forward here. This notebook checks that metadata is present before retrieval and generation.

In [ ]:
def metadata_validation_report(frame: pd.DataFrame, required_fields: List[str]) -> pd.DataFrame:
    rows = []
    for field in required_fields:
        missing_count = int(frame[field].isna().sum() + (frame[field].astype(str).str.strip() == "").sum())
        rows.append({
            "field": field,
            "missing_count": missing_count,
            "missing_percentage": round((missing_count / len(frame)) * 100, 2) if len(frame) else 0.0
        })
    return pd.DataFrame(rows)

validation_report = metadata_validation_report(chunks_df, REQUIRED_METADATA_FIELDS)
validation_report

,field,missing_count,missing_percentage
0,department,0,0.0
1,document_type,0,0.0
2,category,0,0.0
3,region,0,0.0
4,year,0,0.0
5,source,0,0.0
6,file_name,0,0.0


## 6. Load Semantic Embedding Model

In [ ]:
import os
import re
import json
import time
import pickle
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import nest_asyncio

nest_asyncio.apply()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True
)
logger = logging.getLogger("week3_rag")

CONFIG = {
    "repo_root": "/content/hr-compliance-rag",
    "metadata_csv": "/content/hr-compliance-rag/data/metadata.csv",
    "processed_candidates": [
        "/content/hr-compliance-rag/data/processed/chunks.parquet",
        "/content/hr-compliance-rag/data/processed/chunks.csv",
        "/content/hr-compliance-rag/data/processed/processed_chunks.parquet",
        "/content/hr-compliance-rag/data/processed/processed_chunks.csv",
        "/content/metadata_with_chunks.csv" # Added to align with the current notebook's processed_path_used
    ],
    "vectorstore_dir": "/content/hr-compliance-rag/vectorstore",
    "faiss_index_path": "/content/hr-compliance-rag/vectorstore/faiss.index",
    "embeddings_path": "/content/hr-compliance-rag/vectorstore/embeddings.npy",
    "vector_metadata_path": "/content/hr-compliance-rag/vectorstore/metadata_with_chunks.csv",
    "manifest_path": "/content/hr-compliance-rag/vectorstore/manifest.json",
    "evaluation_json_path": "/content/hr-compliance-rag/vectorstore/week3_evaluation_summary.json",
    "evaluation_csv_path": "/content/hr-compliance-rag/vectorstore/week3_query_evaluation.csv",
    "embedding_model_name": "sentence-transformers/all-mpnet-base-v2",
    "reranker_model_name": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "llm_model_name": "google/flan-t5-base",
    "top_k_retrieval": 8,
    "top_k_context": 4,
    "normalize_embeddings": True,
    "max_new_tokens": 220,
    "use_reranker": True,
    "rebuild_index_if_missing": True
}

# Ensure repository structure exists
Path(CONFIG["repo_root"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["vectorstore_dir"]).mkdir(parents=True, exist_ok=True)

print("Repo root:", CONFIG["repo_root"])
print("Vectorstore dir:", CONFIG["vectorstore_dir"])

embedder = SentenceTransformer(CONFIG["embedding_model_name"])
print("Embedding model loaded:", CONFIG["embedding_model_name"])

2026-03-29 09:45:22,851 | INFO | Use pytorch device_name: cpu
2026-03-29 09:45:22,852 | INFO | Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2


Repo root: /content/hr-compliance-rag
Vectorstore dir: /content/hr-compliance-rag/vectorstore


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
2026-03-29 09:45:23,771 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-03-29 09:45:23,779 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-03-29 09:45:23,804 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-b

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

2026-03-29 09:45:23,982 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-29 09:45:23,999 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-03-29 09:45:24,015 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/config_sentence_transformers.json "HTTP/1.1 200 OK"


...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

2026-03-29 09:45:34,549 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-03-29 09:45:34,763 | INFO | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-29 09:45:34,776 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"
2026-03-29 09:45:34,792 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-03-29 09:45:34,905 | INFO | HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-mpnet-base-v2 "HTTP/1.1 200 OK"


Embedding model loaded: sentence-transformers/all-mpnet-base-v2


## 7. Build or Load FAISS Index

Priority:
1. load Week 2 FAISS artifacts if available
2. otherwise rebuild from processed chunks

In [ ]:
def build_embeddings(texts: List[str]) -> np.ndarray:
    embeddings = embedder.encode(
        texts,
        convert_to_numpy=True,
        show_progress_bar=True,
        normalize_embeddings=CONFIG["normalize_embeddings"]
    )
    return embeddings.astype("float32")

def save_manifest(num_vectors: int, embedding_dim: int):
    manifest = {
        "embedding_model_name": CONFIG["embedding_model_name"],
        "llm_model_name": CONFIG["llm_model_name"],
        "reranker_model_name": CONFIG["reranker_model_name"],
        "normalized": bool(CONFIG["normalize_embeddings"]),
        "num_vectors": int(num_vectors),
        "embedding_dimension": int(embedding_dim),
        "processed_source_file": processed_path_used
    }
    with open(CONFIG["manifest_path"], "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

def build_or_load_vectorstore(frame: pd.DataFrame):
    faiss_index_exists = os.path.exists(CONFIG["faiss_index_path"])
    embeddings_exists = os.path.exists(CONFIG["embeddings_path"])
    vector_metadata_exists = os.path.exists(CONFIG["vector_metadata_path"])

    if faiss_index_exists and vector_metadata_exists:
        logger.info("Loading existing FAISS index from Week 2 artifacts")
        index = faiss.read_index(CONFIG["faiss_index_path"])
        vector_meta = pd.read_csv(CONFIG["vector_metadata_path"])
        if embeddings_exists:
            embeddings = np.load(CONFIG["embeddings_path"])
        else:
            embeddings = None
        return index, vector_meta, embeddings, "loaded_existing"

    if not CONFIG["rebuild_index_if_missing"]:
        raise FileNotFoundError("Week 2 vector artifacts not found and rebuilding is disabled.")

    logger.info("Building new FAISS index from processed chunks")
    texts = frame["chunk"].tolist()
    embeddings = build_embeddings(texts)
    embedding_dim = embeddings.shape[1]

    if CONFIG["normalize_embeddings"]:
        index = faiss.IndexFlatIP(embedding_dim)
    else:
        index = faiss.IndexFlatL2(embedding_dim)

    index.add(embeddings)
    faiss.write_index(index, CONFIG["faiss_index_path"])
    np.save(CONFIG["embeddings_path"], embeddings)
    frame.to_csv(CONFIG["vector_metadata_path"], index=False)
    save_manifest(len(frame), embedding_dim)

    return index, frame.copy(), embeddings, "rebuilt"

index, vector_meta_df, all_embeddings, vectorstore_mode = build_or_load_vectorstore(chunks_df)

print("Vectorstore mode:", vectorstore_mode)
print("FAISS vectors:", index.ntotal)
print("Vector metadata rows:", len(vector_meta_df))

2026-03-29 11:01:37,514 | INFO | Loading existing FAISS index from Week 2 artifacts


Vectorstore mode: loaded_existing
FAISS vectors: 2129
Vector metadata rows: 2129


## 8. Semantic Retrieval with Metadata Filtering

In [ ]:
def apply_metadata_filters(frame: pd.DataFrame, filters: Optional[dict] = None) -> pd.DataFrame:
    filtered = frame.copy()
    if not filters:
        return filtered

    for key, value in filters.items():
        if key not in filtered.columns:
            continue
        if isinstance(value, (list, tuple, set)):
            filtered = filtered[filtered[key].isin(value)]
        else:
            filtered = filtered[filtered[key].astype(str) == str(value)]
    return filtered.reset_index(drop=True)

def retrieve_semantic(query: str, top_k: Optional[int] = None, filters: Optional[dict] = None) -> pd.DataFrame:
    top_k = top_k or CONFIG["top_k_retrieval"]
    filtered_df = apply_metadata_filters(vector_meta_df, filters)

    if filtered_df.empty:
        return pd.DataFrame()

    candidate_positions = filtered_df.index.to_list()

    if all_embeddings is None:
        candidate_embeddings = build_embeddings(filtered_df["chunk"].tolist())
    else:
        candidate_embeddings = all_embeddings[candidate_positions]

    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=CONFIG["normalize_embeddings"]
    ).astype("float32")[0]

    if CONFIG["normalize_embeddings"]:
        scores = np.dot(candidate_embeddings, query_embedding)
    else:
        scores = -np.linalg.norm(candidate_embeddings - query_embedding, axis=1)

    top_positions = np.argsort(scores)[::-1][:top_k]
    results = filtered_df.iloc[top_positions].copy().reset_index(drop=True)
    results.insert(0, "score", [float(scores[pos]) for pos in top_positions])
    results.insert(0, "rank", range(1, len(results) + 1))
    return results

## 9. Optional Reranking for Better Accuracy

This stage improves semantic retrieval quality by rescoring query-chunk pairs before context injection.

In [ ]:
try:
    from sentence_transformers import CrossEncoder
    reranker = CrossEncoder(CONFIG["reranker_model_name"]) if CONFIG["use_reranker"] else None
    print("Reranker loaded:", CONFIG["use_reranker"])
except Exception as e:
    reranker = None
    print("Reranker unavailable, continuing without it:", e)

def rerank_results(query: str, results_df: pd.DataFrame, final_k: Optional[int] = None) -> pd.DataFrame:
    final_k = final_k or CONFIG["top_k_context"]

    if results_df.empty:
        return results_df

    if reranker is None:
        return results_df.head(final_k).copy()

    pairs = [[query, text] for text in results_df["chunk"].tolist()]
    rerank_scores = reranker.predict(pairs)

    reranked = results_df.copy()
    reranked["rerank_score"] = rerank_scores
    reranked = reranked.sort_values("rerank_score", ascending=False).reset_index(drop=True)
    reranked["final_rank"] = range(1, len(reranked) + 1)
    return reranked.head(final_k).copy()

2026-03-29 11:01:54,340 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-29 11:01:54,427 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-29 11:01:54,440 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cross-encoder/ms-marco-MiniLM-L6-v2/c5ee24cb16019beea0893ab7796b1df96625c6b8/config.json "HTTP/1.1 200 OK"
2026-03-29 11:01:54,459 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/cross-encoder/ms-marco-MiniLM-L6-v2/c5ee24cb16019beea0893ab7796b1df96625c6b8/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

2026-03-29 11:01:54,596 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/adapter_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-29 11:01:54,684 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-03-29 11:01:54,829 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-29 11:01:54,914 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-29 11:01:54,927 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cross-encoder/ms-marco-MiniLM-L6-v2/c5ee24cb16019beea0893ab7796b1df96625c6b8/config.json "HTTP/1.1 200 OK"
2026-03-29 11:01:55,019 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

2026-03-29 11:01:58,628 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/chat_template.jinja "HTTP/1.1 307 Temporary Redirect"
2026-03-29 11:01:58,719 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-03-29 11:01:58,967 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-29 11:01:59,052 | INFO | HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-29 11:01:59,066 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cross-encoder/ms-marco-MiniLM-L6-v2/c5ee24cb16019beea0893ab7796b1df96625c6b8/README.md "HTTP/1.1 200 OK"
2026-03-29 11:01:59,084 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/cross-encod

README.md: 0.00B [00:00, ?B/s]

2026-03-29 11:01:59,119 | INFO | Use pytorch device: cpu
2026-03-29 11:01:59,214 | INFO | HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L-6-v2 "HTTP/1.1 307 Temporary Redirect"
2026-03-29 11:01:59,304 | INFO | HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"


Reranker loaded: True


In [ ]:
test_query = "What are the employee handbook and compliance policy requirements?"
initial_results = retrieve_semantic(test_query, top_k=CONFIG["top_k_retrieval"])
final_results = rerank_results(test_query, initial_results, final_k=CONFIG["top_k_context"])

print("Initial retrieved results:")
display(initial_results.head(5))

print("Final reranked results:")
display(final_results)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Initial retrieved results:


,rank,score,chunk_id,chunk,chunk_tokens,department,document_type,category,region,year,source,file_name
0,1,0.406810,eurlex_371_chunk_5,to which he is assigned. Authorisation shall b...,900,Human Resources,Policy/Legal,Compliance,Global,2021,eurlex,eurlex_371.txt
1,2,0.394534,eurlex_91_chunk_4,the applicable requirements of the internation...,900,Human Resources,Policy/Legal,Compliance,Global,2021,eurlex,eurlex_91.txt
2,3,0.384842,eurlex_54_chunk_3,the ability of the software to handle a given ...,900,Human Resources,Policy/Legal,Compliance,Global,2024,eurlex,eurlex_54.txt
3,4,0.380826,eurlex_352_chunk_1,COMMISSION DECISION of 9 March 1998 on the pro...,692,Human Resources,Policy/Legal,Compliance,Global,2022,eurlex,eurlex_352.txt
4,5,0.377763,eurlex_534_chunk_7,service providers and trade unions and will in...,760,Human Resources,Policy/Legal,Compliance,Global,2024,eurlex,eurlex_534.txt


Final reranked results:


,rank,score,chunk_id,chunk,chunk_tokens,department,document_type,category,region,year,source,file_name,rerank_score,final_rank
0,8,0.371536,eurlex_54_chunk_4,compliance with the requirements set out in An...,703,Human Resources,Policy/Legal,Compliance,Global,2024,eurlex,eurlex_54.txt,-4.358321,1
1,2,0.394534,eurlex_91_chunk_4,the applicable requirements of the internation...,900,Human Resources,Policy/Legal,Compliance,Global,2021,eurlex,eurlex_91.txt,-6.841576,2
2,6,0.373968,eurlex_534_chunk_5,or to prohibit or restrict their being placed ...,900,Human Resources,Policy/Legal,Compliance,Global,2024,eurlex,eurlex_534.txt,-7.160101,3
3,7,0.372736,eurlex_338_chunk_3,5 of the Regulation provides that the notifica...,900,Human Resources,Policy/Legal,Compliance,Global,2023,eurlex,eurlex_338.txt,-7.396952,4


## 10. Grounded Prompt Engineering

In [ ]:
@dataclass
class RetrievedChunk:
    chunk_id: str
    text: str
    source: str
    metadata: Dict[str, Any]
    score: float

def dataframe_to_chunks(results_df: pd.DataFrame) -> List[RetrievedChunk]:
    chunks = []
    if results_df.empty:
        return chunks

    for _, row in results_df.iterrows():
        meta = {
            "department": row.get("department"),
            "document_type": row.get("document_type"),
            "category": row.get("category"),
            "region": row.get("region"),
            "year": row.get("year"),
            "source": row.get("source"),
            "file_name": row.get("file_name")
        }
        score_val = row["rerank_score"] if "rerank_score" in results_df.columns else row["score"]
        chunks.append(
            RetrievedChunk(
                chunk_id=str(row["chunk_id"]),
                text=str(row["chunk"]),
                source=str(row["source"]),
                metadata=meta,
                score=float(score_val)
            )
        )
    return chunks

def format_context(chunks: List[RetrievedChunk]) -> str:
    parts = []
    for i, chunk in enumerate(chunks, start=1):
        parts.append(
            f"[Context {i}]\n"
            f"Chunk ID: {chunk.chunk_id}\n"
            f"Source: {chunk.source}\n"
            f"Metadata: {json.dumps(chunk.metadata, ensure_ascii=False)}\n"
            f"Text: {chunk.text}\n"
        )
    return "\n".join(parts)

def build_prompt(question: str, chunks: List[RetrievedChunk]) -> str:
    context_block = format_context(chunks)
    return f'''
You are a retrieval-grounded HR and compliance assistant.

Answer the user's question using ONLY the retrieved context.
Do not use outside knowledge.
Do not guess.
If the context is insufficient, say:
"I do not have enough information in the retrieved documents to answer that."

Rules:
1. Stay grounded in the provided context only.
2. Keep the answer clear, factual, and concise.
3. Mention support using citations in this format:
   [Source: <source>, Chunk: <chunk_id>]
4. Preserve metadata traceability in your reasoning.
5. Do not bypass retrieval.

User Question:
{question}

Retrieved Context:
{context_block}

Grounded Answer:
'''.strip()

## 11. Load LLM

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

try:
    # Try to load the model using the pipeline as intended
    llm = pipeline(
        "text2text-generation",
        model=CONFIG["llm_model_name"],
        tokenizer=CONFIG["llm_model_name"]
    )
    print("LLM loaded via pipeline:", CONFIG["llm_model_name"])
except KeyError as e:
    logger.warning(f"text2text-generation pipeline task failed: {e}. Falling back to direct model loading.")

    # Fallback: Load tokenizer and model directly and create a custom generation function
    tokenizer = AutoTokenizer.from_pretrained(CONFIG["llm_model_name"])
    model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["llm_model_name"])

    def custom_llm_generate(prompt, max_new_tokens, do_sample):
        inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample)
        decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return [{"generated_text": decoded_output}]

    llm = custom_llm_generate
    print("LLM loaded via direct model loading:", CONFIG["llm_model_name"])


2026-03-29 11:02:33,040 | INFO | HTTP Request: HEAD https://huggingface.co/google/flan-t5-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-29 11:02:33,055 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-base/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2/config.json "HTTP/1.1 200 OK"
2026-03-29 11:02:33,070 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/flan-t5-base/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2/config.json "HTTP/1.1 200 OK"


config.json: 0.00B [00:00, ?B/s]

2026-03-29 11:02:33,181 | INFO | HTTP Request: HEAD https://huggingface.co/google/flan-t5-base/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-03-29 11:02:33,189 | WARNING | text2text-generation pipeline task failed: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX

tokenizer_config.json: 0.00B [00:00, ?B/s]

2026-03-29 11:02:33,558 | INFO | HTTP Request: GET https://huggingface.co/api/models/google/flan-t5-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-29 11:02:33,661 | INFO | HTTP Request: GET https://huggingface.co/api/models/google/flan-t5-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-03-29 11:02:33,753 | INFO | HTTP Request: HEAD https://huggingface.co/google/flan-t5-base/resolve/main/spiece.model "HTTP/1.1 302 Found"
2026-03-29 11:02:33,842 | INFO | HTTP Request: GET https://huggingface.co/api/models/google/flan-t5-base/xet-read-token/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2 "HTTP/1.1 200 OK"


...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
2026-03-29 11:02:52,767 | INFO | HTTP Request: HEAD https://huggingface.co/google/flan-t5-base/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-29 11:02:52,785 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/flan-t5-base/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2/generation_config.json "HTTP/1.1 200 OK"
2026-03-29 11:02:52,804 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/flan-t5-base/7bcac572ce56db69c1ea7c8af255c5d7c9672fc2/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM loaded via direct model loading: google/flan-t5-base


## 12. End-to-End RAG Pipeline

In [ ]:
def extract_citations(chunks: List[RetrievedChunk]) -> List[Dict[str, Any]]:
    return [
        {
            "source": c.source,
            "chunk_id": c.chunk_id,
            "metadata": c.metadata,
            "score": c.score
        }
        for c in chunks
    ]

def generate_grounded_answer(
    question: str,
    top_k_retrieval: Optional[int] = None,
    top_k_context: Optional[int] = None,
    filters: Optional[dict] = None
) -> Dict[str, Any]:
    start = time.perf_counter()

    retrieval_k = top_k_retrieval or CONFIG["top_k_retrieval"]
    context_k = top_k_context or CONFIG["top_k_context"]

    retrieved_df = retrieve_semantic(question, top_k=retrieval_k, filters=filters)
    final_df = rerank_results(question, retrieved_df, final_k=context_k)
    chunks = dataframe_to_chunks(final_df)

    if not chunks:
        return {
            "question": question,
            "answer": "No relevant context was retrieved.",
            "citations": [],
            "latency_seconds": round(time.perf_counter() - start, 3),
            "retrieved_count": 0
        }

    prompt = build_prompt(question, chunks)

    generation = llm(
        prompt,
        max_new_tokens=CONFIG["max_new_tokens"],
        do_sample=False
    )

    answer = generation[0]["generated_text"].strip()
    latency = round(time.perf_counter() - start, 3)

    return {
        "question": question,
        "answer": answer,
        "citations": extract_citations(chunks),
        "latency_seconds": latency,
        "retrieved_count": len(chunks)
    }

In [ ]:
sample_question = "What do the retrieved HR or compliance documents say about policy requirements?"
sample_output = generate_grounded_answer(sample_question)
sample_output


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{'question': 'What do the retrieved HR or compliance documents say about policy requirements?',
 'answer': 'I do not have enough information in the retrieved documents to answer that.',
 'citations': [{'source': 'eurlex',
   'chunk_id': 'eurlex_25_chunk_9',
   'metadata': {'department': 'Human Resources',
    'document_type': 'Policy/Legal',
    'category': 'Compliance',
    'region': 'Global',
    'year': 2020,
    'source': 'eurlex',
    'file_name': 'eurlex_25.txt'},
   'score': -7.841340065002441},
  {'source': 'eurlex',
   'chunk_id': 'eurlex_25_chunk_10',
   'metadata': {'department': 'Human Resources',
    'document_type': 'Policy/Legal',
    'category': 'Compliance',
    'region': 'Global',
    'year': 2020,
    'source': 'eurlex',
    'file_name': 'eurlex_25.txt'},
   'score': -7.95964241027832},
  {'source': 'eurlex',
   'chunk_id': 'eurlex_545_chunk_2',
   'metadata': {'department': 'Human Resources',
    'document_type': 'Policy/Legal',
    'category': 'Compliance',
    're

## 13. FastAPI API Development

In [ ]:
class QueryRequest(BaseModel):
    question: str = Field(..., min_length=3, description="User question")
    top_k_retrieval: Optional[int] = Field(default=8, ge=1, le=20)
    top_k_context: Optional[int] = Field(default=4, ge=1, le=10)
    filters: Optional[dict] = None

class QueryResponse(BaseModel):
    question: str
    answer: str
    citations: List[Dict[str, Any]]
    latency_seconds: float
    retrieved_count: int

app = FastAPI(
    title="Week 3 HR Compliance RAG API",
    description="Grounded semantic RAG with metadata-aware retrieval and FastAPI",
    version="1.0.0"
)

@app.get("/health")
def health():
    return {
        "status": "ok",
        "embedding_model": CONFIG["embedding_model_name"],
        "llm_model": CONFIG["llm_model_name"],
        "reranker_enabled": CONFIG["use_reranker"],
        "vectors_available": int(index.ntotal),
        "vectorstore_mode": vectorstore_mode
    }

@app.post("/query", response_model=QueryResponse)
def query_endpoint(payload: QueryRequest):
    try:
        result = generate_grounded_answer(
            question=payload.question,
            top_k_retrieval=payload.top_k_retrieval,
            top_k_context=payload.top_k_context,
            filters=payload.filters
        )
        return QueryResponse(**result)
    except Exception as e:
        logger.exception("Query endpoint failed")
        raise HTTPException(status_code=500, detail=str(e))

In [ ]:
import uvicorn
import asyncio
from uvicorn.config import Config

# The 'app' object is defined in cell e8234825

# Create Uvicorn Config
config = Config(app, host="0.0.0.0", port=8000)

# Create Uvicorn Server instance
server = uvicorn.Server(config)

# Get the current running event loop (made available by nest_asyncio.apply())
loop = asyncio.get_event_loop()

# Schedule the server's serve coroutine as a task in the event loop.
# This makes the server run in the background.
loop.create_task(server.serve())

print("Uvicorn server started in the background on http://0.0.0.0:8000")
print("You can now send requests to the API.")
# Note: To stop the server gracefully, you might need to interrupt the kernel or manage the task explicitly.

Uvicorn server started in the background on http://0.0.0.0:8000
You can now send requests to the API.


## 14. Evaluation and Testing

In [ ]:
test_questions = [
    "What kind of HR policies are present in the retrieved system?",
    "How does metadata traceability help the answer?",
    "What should the system do if context is insufficient?",
    "What endpoints are available in the API?"
]

rows = []
for q in test_questions:
    result = generate_grounded_answer(q)
    rows.append({
        "question": q,
        "answer_preview": result["answer"][:350],
        "retrieved_count": result["retrieved_count"],
        "latency_seconds": result["latency_seconds"],
        "citation_count": len(result["citations"])
    })

eval_df = pd.DataFrame(rows)
eval_df

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,question,answer_preview,retrieved_count,latency_seconds,citation_count
0,What kind of HR policies are present in the re...,Compliance,4,12.300,4
1,How does metadata traceability help the answer?,This data enables the software life cycle proc...,4,7.449,4
2,What should the system do if context is insuff...,"The representative of the State has a duty, su...",4,7.421,4
3,What endpoints are available in the API?,I do not have enough information in the retrie...,4,6.745,4


In [ ]:
evaluation_summary = {
    "processed_file_used": processed_path_used,
    "vectorstore_mode": vectorstore_mode,
    "embedding_model_name": CONFIG["embedding_model_name"],
    "reranker_model_name": CONFIG["reranker_model_name"],
    "llm_model_name": CONFIG["llm_model_name"],
    "metadata_validation": validation_report.to_dict(orient="records"),
    "query_count": int(len(eval_df)),
    "avg_latency_seconds": float(eval_df["latency_seconds"].mean()) if len(eval_df) else None,
    "avg_citation_count": float(eval_df["citation_count"].mean()) if len(eval_df) else None
}

with open(CONFIG["evaluation_json_path"], "w", encoding="utf-8") as f:
    json.dump(evaluation_summary, f, indent=2)

eval_df.to_csv(CONFIG["evaluation_csv_path"], index=False)

print("Saved:")
print("-", CONFIG["evaluation_json_path"])
print("-", CONFIG["evaluation_csv_path"])
evaluation_summary

Saved:
- /content/hr-compliance-rag/vectorstore/week3_evaluation_summary.json
- /content/hr-compliance-rag/vectorstore/week3_query_evaluation.csv


{'processed_file_used': '/content/metadata_with_chunks.csv',
 'vectorstore_mode': 'loaded_existing',
 'embedding_model_name': 'sentence-transformers/all-mpnet-base-v2',
 'reranker_model_name': 'cross-encoder/ms-marco-MiniLM-L-6-v2',
 'llm_model_name': 'google/flan-t5-base',
 'metadata_validation': [{'field': 'department',
   'missing_count': 0,
   'missing_percentage': 0.0},
  {'field': 'document_type', 'missing_count': 0, 'missing_percentage': 0.0},
  {'field': 'category', 'missing_count': 0, 'missing_percentage': 0.0},
  {'field': 'region', 'missing_count': 0, 'missing_percentage': 0.0},
  {'field': 'year', 'missing_count': 0, 'missing_percentage': 0.0},
  {'field': 'source', 'missing_count': 0, 'missing_percentage': 0.0},
  {'field': 'file_name', 'missing_count': 0, 'missing_percentage': 0.0}],
 'query_count': 4,
 'avg_latency_seconds': 8.47875,
 'avg_citation_count': 4.0}

## 15. Example API Usage
```python
import requests

requests.get("http://127.0.0.1:8000/health").json()

payload = {
    "question": "Summarize the policy guidance from the retrieved HR documents",
    "top_k_retrieval": 8,
    "top_k_context": 4,
    "filters": {"category": "HR Policies"}
}
requests.post("http://127.0.0.1:8000/query", json=payload).json()
```

## 16. The output retrivals of Week - 3
## which combines the necessary datas or informations from the previous files or tasks for the following benefts as below:

- better semantic retrieval model
- optional reranking for stronger context selection
- metadata-aware filtering
- strict response grounding
- reusable vectorstore logic
- evaluation output saved to the repository structure

This makes it more suitable for stronger Week 3 results while still respecting the Week 1 and Week 2 foundations.

## Note:   
The API semantic code is provided above this code , This can be used for the deployment if the progess is suitable for deployment.